# 01 - Data exploration

Purpose: understand the observation table before anything is built from it.

Questions this notebook answers:

1. What is the coverage of the dataset (rows, districts, date range, gaps)?
2. How does rainfall distribute, and how intermittent is it?
3. Is the dataset labelled synthetic?
4. Is the seasonal signal present and does it shift with latitude?

The generators and loaders come from `rainml`; this notebook only inspects and plots.

In [ ]:
import sys
from pathlib import Path

# The notebook kernel runs with the notebook directory as the working directory,
# but paths are resolved from the repository root so cells behave the same when
# executed by nbconvert, JupyterLab or an IDE.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "ml" / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

ML_SRC = REPO_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = REPO_ROOT / "data" / "processed"
ARTIFACT_ROOT = REPO_ROOT / "ml" / "artifacts"
CONFIG_PATH = REPO_ROOT / "ml" / "configs" / "training.yaml"
OBSERVATIONS = DATA_DIR / "observations.csv"
DISTRICTS = DATA_DIR / "districts.csv"

print("notebook environment ready")
print("repo root:", REPO_ROOT)
print("python", sys.version.split()[0])


In [ ]:
from rainml.data.loaders import load_districts, load_observations, attach_coordinates, daily_completeness, summarise

raw = load_observations(OBSERVATIONS)
districts = load_districts(DISTRICTS)
observations = attach_coordinates(raw, districts)
observations.head()
len(observations)

## Coverage per district

Gaps matter: a district with missing days will look drier than it is unless the
cleaning stage imputes them, which notebook 02 verifies.

In [ ]:
coverage = daily_completeness(observations)
coverage.sort_values("missing_days", ascending=False).head(12)

## Distribution of daily rainfall

Rainfall is intermittent. A large mass at zero and a long right tail is the normal
shape, not a data quality problem to be removed.

In [ ]:
display(observations["rainfall_mm"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(observations["rainfall_mm"], bins=80, color="#2563eb")
axes[0].set_yscale("log")
axes[0].set_xlabel("daily rainfall (mm)")
axes[0].set_title("Daily rainfall distribution (log y)")
axes[0].axvline(15.0, color="#b91c1c", linestyle="--", label="heavy event threshold")
axes[0].legend()

frame = observations.copy()
frame["doy"] = pd.to_datetime(frame["observed_at"]).dt.dayofyear
frame["month"] = pd.to_datetime(frame["observed_at"]).dt.month
monthly = frame.groupby("month")["rainfall_mm"].mean()
axes[1].bar(monthly.index, monthly.values, color="#0f766e")
axes[1].set_xlabel("month")
axes[1].set_ylabel("mean rainfall (mm)")
axes[1].set_title("Mean monthly rainfall")
plt.tight_layout()
plt.show()

## Seasonality versus latitude

Indian rainfall peaks later further north. If that structure is absent, the feature
set in notebook 03 has less to learn from and the geographic features matter more.

In [ ]:
frame["lat_band"] = pd.cut(frame["lat"], bins=[8, 16, 22, 28, 36])
pivot = frame.pivot_table(index="month", columns="lat_band", values="rainfall_mm", aggfunc="mean")
pivot.plot(figsize=(11, 4), marker="o")
plt.ylabel("mean rainfall (mm)")
plt.title("Monthly mean rainfall by latitude band")
plt.tight_layout()
plt.show()
pivot.round(2)

## Summary

Every number below comes from `rainml.data.loaders.summarise`.

In [ ]:
summary = summarise(observations)
summary
pd.DataFrame([summary])

## Provenance check

This dataset is generated by `scripts/generate_demo_dataset.py` and is **synthetic**.
It is used so the pipeline is runnable on a clean checkout. Replace
`data/processed/observations.csv` with a real source before making any claim about
forecast accuracy.

In [ ]:
import json

manifest = DATA_DIR / "README.md"
print(manifest.read_text()[:600] if manifest.exists() else "no manifest")